# Projeto: Normalização e Preparação dos Dados Meteorológicos – Open Weather API


## Descrição Geral
Este notebook trata da normalização e preparação dos dados brutos do Open Weather API, extraídos da camada bronze, para análise e consumo pelas próximas camadas (silver/gold).<br>
Antes de serem processados neste notebook, os dados passam por uma etapa automatizada de validação com Great Expectations (GX). Apenas os registros validados com sucesso são utilizados nas transformações abaixo.<br>
As etapas incluem renomeação de colunas, tratamento de tipos de dados, separação de campos compostos e padronização de formatos.

## Objetivos

- Padronizar os nomes das colunas conforme convenção definida.
- Realizar o ajuste de tipos de dados para colunas de datas, timestamps e decimal.
- Separar colunas compostas (como precipitação e probabilidade).
- Preparar o DataFrame para persistência na camada silver.

## Tecnologias Utilizadas

- Apache Spark (PySpark)
- Databricks Notebooks
- Delta Lake


## Fonte de Dados

- **Localização:** [Tabela open_weather_map](grupo01.bronze.open_weather_map)
- **Formato:** Delta
- **Origem:** Tabela bronze criada a partir da consulta da API Open Weather Map

---


### Histórico de Atualizações

| Data       | Responsável    | Versão | Descrição                                            |
|------------|----------------|--------|------------------------------------------------------|
| 16/05/2025 |  Gabriel Rost  | 1.0    | Pipeline de ingestão do Open Weather map  |
| 19/05/2025 | Evelise Dias   | 1.1    | Atualização de documentação     |
| 21/05/2025 | Lucas Bringmann| 1.2    | Adição de funções necessárias, como desduplicação   |
| 25/05/2025 | Evelise Dias   | 1.3    | Ajuste do pipeline para ler apenas dados validados pelo GX                     |
| 26/05/2025 | Evelise Dias   | 1.4    | Atualização da documentação e sumário para refletir a origem validada dos dados          |


In [0]:
%run ../utils/utils_run_silver  

In [0]:
%run ./queries_silver

In [0]:
pasta_validos = lakehouse_logs_gx + "/bronze/open_weather_map/validos/open_weather_map"
df = ler_tabelas(pasta_validos, fromTable=False)

In [0]:
df = renomear_colunas(df, CONVENCAO_COLUNAS_OPEN_WEATHER_MAP)

In [0]:
df = alterar_tipos(df, ["dt_previsao"], "date")

In [0]:
df = alterar_tipos(df = df,colunas= ["dt_carga"], tipo= "timestamp", data_formato="yyyy-MM-dd HH:mm:ss")

In [0]:
df = alterar_tipos(df = df,colunas= ["umid_min", "umid_max", "temp_min", "temp_max", "precipitacao", "vel_vento"], tipo= "decimal")

In [0]:
spark.sql(create_table_open_weather_map)

In [0]:
df_com_id = gera_primary_key(df)

In [0]:
df_com_id = remover_duplicatas(df_com_id, ["dt_carga", "dt_previsao", "sig_uf","nom_municipio", "flg_fonte_previsao"])

In [0]:
inserir_com_merge(df_com_id, caminho_tabela_open_weather_map_silver, ["id"])